In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
data_path = os.path.join(path, 'Q1_data.csv')
df_data = pd.read_csv(data_path)

print(f"Dataset shape: {df_data.shape}")


In [ ]:
# Task 2: Write your code here:
df_data.head() #showing the first 5

In [ ]:
# Task 3: Write your code here:
df_data.info() #gives us clear info of what we need to know

In [ ]:
# Task 4: Write your code here:
df_data.describe() #describes the scale between data, good for checking the need to scale

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df_data['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.show()

In [ ]:
# Task 1: Write your code here:

df_data.dropna(subset=['Order_ID']) #we do not drop target!

cols = [	'Distance_km',	'Weather',	'Traffic_Level',	'Time_of_Day',	'Vehicle_Type',	'Preparation_Time_min',	'Courier_Experience_yrs']
df_clean = df_data.dropna(subset=cols).copy() #cleaning cols with
print(f"Shape after cleaning: {df_clean.shape}")

In [ ]:
# Task 2: Write your code here:
for col in cols:
    df_clean[col] = df_clean[col].fillna('unknown')#fills missing values



print("Missing values remaining:", df_clean.isnull().sum().sum())

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df_data):
  duplicates = df_data.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df_data.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_data)



In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import LabelEncoder
categorical_cols = df_data.select_dtypes(include=["object"]).columns
label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  df_data[col] = le.fit_transform(df_data[col])
  label_encoders[col] = le

  df_data


In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

numerical_cols = df_data.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df_data[numerical_cols] = scaler.fit_transform(df_data[numerical_cols])
df_data.head()

In [ ]:
# Task 6: Write your code here:



In [ ]:
# Task 1: Write your code here:
X = df_data.drop("Delivery_Time", axis=1).astype(float)
y = df_data['Delivery_Time'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error as sklearn_mse, mean_absolute_error, r2_score
from sklearn.linear_model import Ridge, Lasso
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor

#here we go the traditional way of k-fold
def mean_squared_error(y, y_hat):
  return (1 / (2 * len(y))) * np.sum((y_hat - y) ** 2)



  #defining a gradient descent func
def gradient_descent(X, y, learning_rate, n_iters=500):
  m, n = X.shape  # m rows, n columns
  theta = np.zeros(n)  # initialize a zeros weight vector with n dimensions
  losses = []

#here we train the model using everything we did previously
for _ in df_data(range(5000), desc="Training Linear Regression"):
   y_hat = np.dot(X, theta)
   gradient = np.dot(X.T, (y_hat - y)) / m
   theta -= 0.001 * gradient

   loss = mean_squared_error(y, y_hat)
   losses.append(loss)

  return theta, losses

lr_losses = []
lr_mse = []
lr_rmse = []
lr_r2 = []


n_splits = 5

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train
  theta, losses = gradient_descent(X_train.values, y_train.values, learning_rate=0.1, n_iters=500)

  # Validate
  y_pred = np.dot(X_test.values, theta)

  # Calculate evaluation metrics
  mse = sklearn_mse(y_test, y_pred)
  rmse = np.sqrt(mse)
  r2 = r2_score(y_test, y_pred)

  # Store results
  lr_losses.append(losses)
  lr_mse.append(mse)
  lr_rmse.append(rmse)
  lr_r2.append(r2)







In [ ]:
# Task 1: Write your code here:

models = {
  "Ridge Regression": Ridge(alpha=1.0, max_iter=10000),
  "LASSO Regression": Lasso(alpha=1.0,  max_iter=10000),
  "Support Vector Machine": SVR(kernel='rbf'),
  "Decision Tree Regressor": DecisionTreeRegressor(max_depth=10),
  "Random Forest Regressor": RandomForestRegressor(n_estimators=200),
  "LightGBM": LGBMRegressor(verbose=-1),

}
coeffs = {}

coeffs['Lasso'] = models['LASSO Regression'].coef_
coeffs['Ridge'] = models['Ridge Regression'].coef_

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, coef) in enumerate(coeffs.items()):
  # Sort features by absolute coefficient value
  absolute_coef = np.abs(coef)
  sorted_idx = np.argsort(absolute_coef)

  ax = axes[i]
  ax.barh(features[sorted_idx], coef[sorted_idx])
  ax.set_title(f"{model_name} Coefficients")
  ax.set_xlabel("Coefficient Value (Impact)")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:


In [ ]:
# Task Bonus: Write your code here: